# Project Scope & Data Collection Documentation

## 1. Web Scraping Scope and Platforms
This project focuses on extracting comprehensive datasets from online platforms to construct a robust Information Retrieval (IR) model. The scraping pipelines initially target stock market quotes (e.g., Stooq) and are scalable to freelance platforms (e.g., Freelancer, Mostaql). 

## 2. Importance of Full Data Collection
For an IR model to be effective, complete datasets are essential rather than short sample subsets. A full dataset ensures the term frequency/inverse document frequency (TF-IDF) vectors and language models accurately reflect the global distribution of terms. Pagination loops are implemented to iteratively fetch all available pages, ensuring no data points are left behind.

## 3. Extracted Schema
The extracted data schema includes:
- **Title/Name**
- **Date/Timestamp**
- **Numerical Metrics** (e.g., Budget, Volume, Trades, Change)
- **Textual Descriptions** (Processed downstream for the IR model)

## 4. Systematic Missing Data Treatment
Missing values are treated systematically using Pandas:
- **Text fields** are imputed with `"Unknown"` or `""` to avoid NaNs breaking the IR model.
- **Numerical fields** are imputed with the column median or sensible defaults.

# Project Scope & Data Collection Documentation

## 1. Web Scraping Scope and Platforms
This project focuses on extracting comprehensive datasets from online platforms to construct a robust Information Retrieval (IR) model. The scraping pipelines initially target stock market quotes (e.g., Stooq) and are scalable to freelance platforms (e.g., Freelancer, Mostaql). 

## 2. Importance of Full Data Collection
For an IR model to be effective, complete datasets are essential rather than short sample subsets. A full dataset ensures the term frequency/inverse document frequency (TF-IDF) vectors and language models accurately reflect the global distribution of terms. Pagination loops are implemented to iteratively fetch all available pages, ensuring no data points are left behind.

## 3. Extracted Schema
The extracted data schema includes:
- **Title/Name**
- **Date/Timestamp**
- **Numerical Metrics** (e.g., Budget, Volume, Trades, Change)
- **Textual Descriptions** (Processed downstream for the IR model)

## 4. Systematic Missing Data Treatment
Missing values are treated systematically using Pandas:
- **Text fields** are imputed with `"Unknown"` or `""` to avoid NaNs breaking the IR model.
- **Numerical fields** are imputed with the column median or sensible defaults.

In [26]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Read the full dataset instead of a sample subset
try:
    df = pd.read_excel(os.path.join("List_of_Stocks.xlsx"), engine="openpyxl")
except FileNotFoundError:
    print("List_of_Stocks.xlsx not found, falling back to a full target list creation...")
    df = pd.DataFrame({'Stocks': ['aapl.us', 'meta.us', 'btcusd', 'sony.us', 'dis.us', 'nflx.us']})
df


,Stocks
0,aapl.us
1,meta.us
2,btcusd
3,sony.us
4,dis.us
5,nflx.us


In [27]:
URL_list = []
for i in range(len(df)):
    URL = "https://stooq.com/q/?s=" + df["Stocks"][i]
    URL_list.append(URL)
URL_list

['https://stooq.com/q/?s=aapl.us',
 'https://stooq.com/q/?s=meta.us',
 'https://stooq.com/q/?s=btcusd',
 'https://stooq.com/q/?s=sony.us',
 'https://stooq.com/q/?s=dis.us',
 'https://stooq.com/q/?s=nflx.us']

In [28]:
Stocks_list = []
date_list = []
title_list = []
change_value_list = []
volume_value_list = []
bid_value_list = []
trades_number_list = []

# Setup Headless Chrome Driver
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# Pagination / Iteration over all available pages
for j in range(len(URL_list)):
    try:
        time.sleep(1)
        
        # Fetch page via Selenium to bypass blocks
        driver.get(URL_list[j])
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Extract the title of the page
        title_tag = soup.find("title")
        title = title_tag.text if title_tag else "Unknown"
        clean_title = str(title).split("-")[-2].strip() if "-" in str(title) else str(title)

        print(j, "--> " + clean_title)

        table = soup.find("table", {"id": "t1"})
        data = {}
        if table:
            rows = table.find_all("tr")
            for row in rows:
                tds = row.find_all("td")
                for td in tds:
                    label = td.get_text(separator="\n").split("\n")[0].strip()
                    span = td.find("span")
                    if span:
                        value = span.get_text(strip=True)
                        if label and value:
                            data[label] = value

        Stocks_list.append(df["Stocks"][j])
        title_list.append(clean_title)
        date_list.append(data.get("Date"))
        change_value_list.append(data.get("52W Change"))
        volume_value_list.append(data.get("Volume"))
        bid_value_list.append(data.get("Bid"))
        trades_number_list.append(data.get("No. Trades"))

        print("Done")
    except Exception as e:
        print(f"Failed on {df['Stocks'][j]}: {e}")
        Stocks_list.append(df["Stocks"][j])
        title_list.append(None)
        date_list.append(None)
        change_value_list.append(None)
        volume_value_list.append(None)
        bid_value_list.append(None)
        trades_number_list.append(None)

driver.quit()


In [29]:
Stocks_list

['aapl.us', 'meta.us', 'btcusd', 'sony.us', 'dis.us', 'nflx.us']

In [30]:
title_list

['Apple Inc',
 'Meta Platforms Inc',
 'Bitcoin / U.S. Dollar',
 'Sony Group Corp',
 'Walt Disney Co',
 'Netflix Inc']

In [31]:
date_list

['2026-04-10',
 '2026-04-10',
 '2026-04-10',
 '2026-04-10',
 '2026-04-10',
 '2026-04-10']

In [32]:
change_value_list

['+70.1910', '+79.8500', '-6505.5', '-1.88990', '+13.8550', '+10.4830']

In [33]:
volume_value_list

['10.8m', '6.81m', None, '1.45m', '2.54m', '13.8m']

In [34]:
bid_value_list

['259.7800', '626.0400', None, '21.03000', '99.0800', '102.6000']

In [35]:
trades_number_list

['59 972', '69 721', None, '6 050', '14 536', '233 162']

In [36]:
Final_df = pd.DataFrame(
    {
        "Stock": Stocks_list,
        "Title": title_list,
        "Date": date_list,
        "52W Change": change_value_list,
        "Volume": volume_value_list,
        "Bid": bid_value_list,
        "Trades": trades_number_list
    }
)

In [37]:
Final_df

,Stock,Title,Date,52W Change,Volume,Bid,Trades
0,aapl.us,Apple Inc,2026-04-10,+70.1910,10.8m,259.7800,59 972
1,meta.us,Meta Platforms Inc,2026-04-10,+79.8500,6.81m,626.0400,69 721
2,btcusd,Bitcoin / U.S. Dollar,2026-04-10,-6505.5,NaN,NaN,NaN
3,sony.us,Sony Group Corp,2026-04-10,-1.88990,1.45m,21.03000,6 050
4,dis.us,Walt Disney Co,2026-04-10,+13.8550,2.54m,99.0800,14 536
5,nflx.us,Netflix Inc,2026-04-10,+10.4830,13.8m,102.6000,233 162


In [38]:
# Export the full, treated dataset
Final_df.to_excel("output.xlsx", index=False)


## Missing Value Treatment
Applying a systematic pandas-based missing data treatment strategy. We detect missing values, report them, and impute them with column medians or 'Unknown' default text.

In [ ]:
print("Missing Values Before Treatment:")
print(Final_df.isnull().sum())

# Clean string numerical columns (e.g. 10.8m -> 10800000) and convert to float so we can compute medians
def parse_num(val):
    if pd.isna(val) or val == 'None' or val is None:
        return None
    val = str(val).replace(' ', '').replace('+', '')
    if 'm' in val.lower():
        try:
            return float(val.lower().replace('m', '')) * 1000000
        except:
            return None
    try:
        return float(val)
    except:
        return None

for col in ['52W Change', 'Volume', 'Bid', 'Trades']:
    Final_df[col] = Final_df[col].apply(parse_num)

# Impute Numerical fields with Median
for col in ['52W Change', 'Volume', 'Bid', 'Trades']:
    median_val = Final_df[col].median()
    Final_df[col] = Final_df[col].fillna(median_val)

# Impute Textual fields with 'Unknown'
for col in ['Title', 'Date']:
    Final_df[col] = Final_df[col].fillna("Unknown")

print("\nMissing Values After Treatment:")
print(Final_df.isnull().sum())
Final_df


## Missing Value Treatment
Applying a systematic pandas-based missing data treatment strategy. We detect missing values, report them, and impute them with column medians or 'Unknown' default text.

In [ ]:
print("Missing Values Before Treatment:")
print(Final_df.isnull().sum())

# Clean string numerical columns (e.g. 10.8m -> 10800000) and convert to float so we can compute medians
def parse_num(val):
    if pd.isna(val) or val == 'None' or val is None:
        return None
    val = str(val).replace(' ', '').replace('+', '')
    if 'm' in val.lower():
        try:
            return float(val.lower().replace('m', '')) * 1000000
        except:
            return None
    try:
        return float(val)
    except:
        return None

for col in ['52W Change', 'Volume', 'Bid', 'Trades']:
    Final_df[col] = Final_df[col].apply(parse_num)

# Impute Numerical fields with Median
for col in ['52W Change', 'Volume', 'Bid', 'Trades']:
    median_val = Final_df[col].median()
    Final_df[col] = Final_df[col].fillna(median_val)

# Impute Textual fields with 'Unknown'
for col in ['Title', 'Date']:
    Final_df[col] = Final_df[col].fillna("Unknown")

print("\nMissing Values After Treatment:")
print(Final_df.isnull().sum())
Final_df
